# 25 — Prepare LIANA inputs (canonical label fix)

**Purpose:** Prepare cell groups and metadata inputs for LIANA while preserving the historical
cell-selection semantics and correcting an accidental non-malignant label-prefix bug.

**Historical source notebook:** `23_prepare_LIANA_CCC_inputs_HVG8000_r0_6.ipynb`

## Canonicalization performed here

The historical notebook used:

```python
if "malignant" in comp_low:
    ...
if "non_malignant" in comp_low or "diploid" in comp_low:
    ...
```

Because the string `non_malignant` itself contains `malignant`, the second branch was never
reached for non-malignant/diploid-leaning cells. As a result, historical outputs contained labels
such as `Malignant_non_malignant_oligodendrocyte...`.

This repository notebook corrects **label semantics only** by testing the non-malignant/diploid
branch before the malignant branch. It also writes an explicit historical-to-canonical label audit
and asserts that the set of cells included/excluded from the main and exploratory analyses is
unchanged.

No expression values, cell identities, malignant-state assignments, thresholds, or statistical
results are recomputed here.

Run from top to bottom in a fresh kernel.


# 42 — Prepare LIANA cell–cell communication inputs  
**Project:** GBM scRNA-seq, HVG8000, scVI, CopyKAT consensus  
**Goal:** prepare robust, thesis-ready inputs for LIANA/LIANA+ ligand–receptor analysis.

This notebook does **not** run LIANA yet. It prepares and audits:
- final cell-type labels for communication analysis,
- main vs exploratory CCC grouping,
- per-condition and per-sample cell-type counts,
- inclusion/exclusion decisions,
- lightweight metadata tables for the next LIANA notebook.

Why a preparation step?  
Cell–cell communication inference is sensitive to cell labels, rare groups, review/ambiguous populations, and sample imbalance. Here we define a conservative **main LIANA analysis** and a separate **exploratory review-inclusive analysis**.

In [1]:
import os
import sys
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import scanpy as sc

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed" / "integrated"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURE_DIR = PROJECT_DIR / "figures" / "LIANA_HVG8000_r0_6_prepare"

METADATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

OUT_PREFIX = "LIANA_prepare_HVG8000_r0_6"

INPUT_H5AD = PROCESSED_DIR / "GBM_core_4datasets_final_annotation_malignant_states.h5ad"

if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        f"Required final malignant-state object not found: {INPUT_H5AD}. "
        "Run notebooks 17 and 20 first."
    )

MIN_TOTAL_CELLS_MAIN = 200
MIN_SAMPLES_MAIN = 3
MIN_CELLS_PER_CONDITION_FOR_CONDITION_ANALYSIS = 50

# To save disk space, this notebook does NOT write a duplicate h5ad by default.
# The next LIANA notebook can load the final h5ad and merge the exported metadata.
WRITE_LIANA_READY_H5AD = False

print("[INPUT_H5AD]", INPUT_H5AD)
print("[METADATA_DIR]", METADATA_DIR)

[INPUT_H5AD] G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_final_annotation_malignant_states.h5ad
[METADATA_DIR] G:\Repository\glioma-cnv-single-cell-atlas\metadata


In [2]:
def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")
    return path

def choose_first_existing_column(df, candidates, required=True, label="column"):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"Could not find {label}. Tried: {candidates}. Available: {list(df.columns)}")
    return None

def clean_label(x):
    x = str(x)
    x = x.strip()
    x = x.replace("/", "_")
    x = x.replace("-", "_")
    x = x.replace(" ", "_")
    x = x.replace("__", "_")
    return x

def safe_mode(series):
    s = series.dropna().astype(str)
    if len(s) == 0:
        return np.nan
    return s.value_counts().index[0]

In [3]:
print("[READ backed obs/var]")
adata_b = sc.read_h5ad(INPUT_H5AD, backed="r")
obs = adata_b.obs.copy()
var = adata_b.var.copy()
n_obs = adata_b.n_obs
n_vars = adata_b.n_vars

print(adata_b)
print("[obs columns]", len(obs.columns))
print(list(obs.columns)[:80])
print("[var columns]", list(var.columns)[:20])

[READ backed obs/var]
AnnData object with n_obs × n_vars = 351427 × 15176 backed at 'G:\\Repository\\glioma-cnv-single-cell-atlas\\data\\processed\\integrated\\GBM_core_4datasets_final_annotation_malignant_states.h5ad'
    obs: 'sample_id', 'geo_id', 'core_dataset', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'pct_counts_hb', 'dataset_id', 'study_id', 'source_dataset', 'source_file', 'concat_key', 'patient_id', 'condition', 'batch_dataset', 'batch_sample', 'batch_patient', 'original_cell_barcode', 'diagnosis', 'disease_stage', 'tumor_status', 'platform', 'technology', 'study_batch', 'sample_batch', 'scvi_batch', 'analysis_group_major', 'analysis_group_detail', 'ready_for_integration', 'cell_id_original', 'leiden_scvi_0.1', 'leiden_scvi_0.2', 'leiden_scvi_0.3', 'leiden_scvi_0.4', 'leiden_scvi_0.5', 'leiden_scvi_HVG8000_r0_1', 'leiden_scvi_HVG8000_r0_15', 'leiden_scvi_HVG8000_r0_2', 'leiden_scvi_HVG80

In [4]:
# Resolve exact canonical metadata columns.
# Historical execution resolved to these exact columns. The repository workflow
# now requires them explicitly rather than silently falling back to alternatives.

EXPECTED_COLUMNS = {
    "cluster": "leiden_scvi_main",
    "annotation": "final_annotation_refined",
    "compartment": "final_compartment_major",
    "condition": "condition",
    "sample": "sample_id",
    "patient": "patient_id",
    "dataset": "dataset_id",
    "copykat": "copykat_pred_clean",
    "malignant_state": "malignant_state",
}

missing_required = [
    col for col in EXPECTED_COLUMNS.values()
    if col not in obs.columns
]
if missing_required:
    raise KeyError(
        "Canonical LIANA preparation requires the exact expected metadata columns.\n"
        f"Missing: {missing_required}\n"
        f"Available columns: {list(obs.columns)}"
    )

cluster_col = EXPECTED_COLUMNS["cluster"]
annotation_col = EXPECTED_COLUMNS["annotation"]
compartment_col = EXPECTED_COLUMNS["compartment"]
condition_col = EXPECTED_COLUMNS["condition"]
sample_col = EXPECTED_COLUMNS["sample"]
patient_col = EXPECTED_COLUMNS["patient"]
dataset_col = EXPECTED_COLUMNS["dataset"]
copykat_col = EXPECTED_COLUMNS["copykat"]
malignant_state_col = EXPECTED_COLUMNS["malignant_state"]

resolved_cols = pd.DataFrame([
    {"role": role, "column": col}
    for role, col in EXPECTED_COLUMNS.items()
])

write_tsv_replace(
    resolved_cols,
    METADATA_DIR / f"{OUT_PREFIX}_resolved_columns.tsv",
)
display(resolved_cols)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_prepare_HVG8000_r0_6_resolved_columns.tsv shape=(9, 2)


,role,column
0,cluster,leiden_scvi_main
1,annotation,final_annotation_refined
2,compartment,final_compartment_major
3,condition,condition
4,sample,sample_id
5,patient,patient_id
6,dataset,dataset_id
7,copykat,copykat_pred_clean
8,malignant_state,malignant_state


In [5]:
# Standardize metadata values.
obs = obs.copy()
obs["_cell_id"] = obs.index.astype(str)
obs["_cluster"] = obs[cluster_col].astype(str)
obs["_annotation"] = obs[annotation_col].astype(str)
obs["_compartment"] = obs[compartment_col].astype(str)
obs["_condition"] = obs[condition_col].astype(str)
obs["_sample_id"] = obs[sample_col].astype(str)

if patient_col is not None:
    obs["_patient_id"] = obs[patient_col].astype(str)
else:
    obs["_patient_id"] = obs["_sample_id"]

if dataset_col is not None:
    obs["_dataset_id"] = obs[dataset_col].astype(str)
else:
    obs["_dataset_id"] = "unknown_dataset"

if copykat_col is not None:
    obs["_copykat"] = obs[copykat_col].astype(str)
else:
    obs["_copykat"] = "not_available"

if malignant_state_col is not None:
    obs["_malignant_state"] = obs[malignant_state_col].astype(str)
else:
    obs["_malignant_state"] = "not_available"

print(obs[["_cell_id", "_cluster", "_annotation", "_compartment", "_condition", "_sample_id", "_malignant_state"]].head())

                                                      _cell_id _cluster  \
DS1_GSE103224:PJ016_cell000216  DS1_GSE103224:PJ016_cell000216        7   
DS1_GSE103224:PJ016_cell000217  DS1_GSE103224:PJ016_cell000217        7   
DS1_GSE103224:PJ016_cell000218  DS1_GSE103224:PJ016_cell000218        7   
DS1_GSE103224:PJ016_cell000219  DS1_GSE103224:PJ016_cell000219        2   
DS1_GSE103224:PJ016_cell000220  DS1_GSE103224:PJ016_cell000220        7   

                                                                   _annotation  \
DS1_GSE103224:PJ016_cell000216        malignant_AC_like_reactive_CNV_supported   
DS1_GSE103224:PJ016_cell000217        malignant_AC_like_reactive_CNV_supported   
DS1_GSE103224:PJ016_cell000218        malignant_AC_like_reactive_CNV_supported   
DS1_GSE103224:PJ016_cell000219  cycling_malignant_like_candidate_CNV_supported   
DS1_GSE103224:PJ016_cell000220        malignant_AC_like_reactive_CNV_supported   

                                                    _com

In [6]:
# Define LIANA cell-type labels.
#
# Main CCC analysis:
#   malignant states + high-confidence non-malignant/diploid-leaning lineages.
#
# Review/ambiguous populations:
#   excluded from the main analysis but retained in the exploratory label.
#
# IMPORTANT HISTORICAL BUG:
# The historical implementation tested `"malignant" in comp_low` BEFORE
# `"non_malignant" in comp_low`. Because "non_malignant" contains the substring
# "malignant", high-confidence non-malignant cells received misleading labels
# such as `Malignant_non_malignant_...`.
#
# We preserve the historical function below ONLY to audit cell membership.
# The canonical function then fixes the branch ordering.

def assign_liana_label_historical(row, include_review=False):
    cluster = str(row["_cluster"])
    ann = str(row["_annotation"])
    comp = str(row["_compartment"])
    state = str(row["_malignant_state"])
    ann_clean = clean_label(ann)

    comp_low = comp.lower()
    ann_low = ann.lower()
    state_low = state.lower()

    if "cycling" in comp_low or "cycling" in state_low or cluster == "2":
        return "Malignant_cycling_candidate"

    # Historical substring-order bug preserved for audit only.
    if "malignant" in comp_low:
        if state not in ["not_available", "not_malignant_state", "nan", "None"]:
            s = clean_label(state)
            s = s.replace("condition_malignant_state_", "")
            s = s.replace("state_marker_", "")
            return f"Malignant_{s}"
        if cluster == "0":
            return "Malignant_OPC_NPC_like"
        if cluster == "4":
            return "Malignant_hypoxia_stress"
        if cluster == "7":
            return "Malignant_AC_reactive"
        if cluster == "11":
            return "Malignant_NPC_neuronal_like"
        return f"Malignant_{ann_clean}"

    if "non_malignant" in comp_low or "diploid" in comp_low:
        if "oligodendrocyte" in ann_low:
            return "Oligodendrocyte_myelinating"
        if "endothelial" in ann_low:
            return "Endothelial"
        if "mural" in ann_low or "pericyte" in ann_low or "fibroblast" in ann_low:
            return "Mural_Pericyte_Fibroblast"
        if "b_plasma" in ann_low or "plasma" in ann_low or ann_low.startswith("b_"):
            return "B_Plasma"
        if "mast" in ann_low:
            return "Mast_cell"
        return clean_label(ann)

    if include_review:
        return f"Review_cluster_{cluster}_{ann_clean}"

    return np.nan


def assign_liana_label(row, include_review=False):
    """Canonical LIANA label assignment with corrected compartment precedence."""
    cluster = str(row["_cluster"])
    ann = str(row["_annotation"])
    comp = str(row["_compartment"])
    state = str(row["_malignant_state"])
    ann_clean = clean_label(ann)

    comp_low = comp.lower()
    ann_low = ann.lower()
    state_low = state.lower()

    # 1) Cycling malignant candidate has explicit precedence.
    if "cycling" in comp_low or "cycling" in state_low or cluster == "2":
        return "Malignant_cycling_candidate"

    # 2) High-confidence non-malignant / diploid-leaning groups MUST be checked
    #    before the generic malignant substring test.
    if "non_malignant" in comp_low or "diploid" in comp_low:
        if "oligodendrocyte" in ann_low:
            return "Oligodendrocyte_myelinating"
        if "endothelial" in ann_low:
            return "Endothelial"
        if "mural" in ann_low or "pericyte" in ann_low or "fibroblast" in ann_low:
            return "Mural_Pericyte_Fibroblast"
        if "b_plasma" in ann_low or "plasma" in ann_low or ann_low.startswith("b_"):
            return "B_Plasma"
        if "mast" in ann_low:
            return "Mast_cell"
        return clean_label(ann)

    # 3) Malignant states.
    if "malignant" in comp_low:
        if state not in ["not_available", "not_malignant_state", "nan", "None"]:
            s = clean_label(state)
            s = s.replace("condition_malignant_state_", "")
            s = s.replace("state_marker_", "")
            return f"Malignant_{s}"
        if cluster == "0":
            return "Malignant_OPC_NPC_like"
        if cluster == "4":
            return "Malignant_hypoxia_stress"
        if cluster == "7":
            return "Malignant_AC_reactive"
        if cluster == "11":
            return "Malignant_NPC_neuronal_like"
        return f"Malignant_{ann_clean}"

    # 4) Review/ambiguous groups are exploratory only.
    if include_review:
        return f"Review_cluster_{cluster}_{ann_clean}"

    return np.nan


# Historical labels for audit only.
hist_main = obs.apply(
    lambda r: assign_liana_label_historical(r, include_review=False),
    axis=1,
)
hist_expl = obs.apply(
    lambda r: assign_liana_label_historical(r, include_review=True),
    axis=1,
)

# Canonical corrected labels used downstream.
obs["liana_celltype_main"] = obs.apply(
    lambda r: assign_liana_label(r, include_review=False),
    axis=1,
)
obs["liana_celltype_exploratory"] = obs.apply(
    lambda r: assign_liana_label(r, include_review=True),
    axis=1,
)

# Membership must remain identical: this notebook corrects label semantics,
# not which cells participate in LIANA.
if not hist_main.notna().equals(obs["liana_celltype_main"].notna()):
    raise RuntimeError(
        "Canonical label correction changed main-analysis cell membership. "
        "Stop and audit before proceeding."
    )

if not hist_expl.notna().equals(obs["liana_celltype_exploratory"].notna()):
    raise RuntimeError(
        "Canonical label correction changed exploratory-analysis cell membership. "
        "Stop and audit before proceeding."
    )

label_audit = pd.DataFrame({
    "historical_main_label": hist_main,
    "canonical_main_label": obs["liana_celltype_main"],
    "historical_exploratory_label": hist_expl,
    "canonical_exploratory_label": obs["liana_celltype_exploratory"],
})

label_audit_summary = (
    label_audit
    .groupby(
        [
            "historical_main_label",
            "canonical_main_label",
            "historical_exploratory_label",
            "canonical_exploratory_label",
        ],
        dropna=False,
        observed=True,
    )
    .size()
    .reset_index(name="n_cells")
    .sort_values("n_cells", ascending=False)
)

write_tsv_replace(
    label_audit_summary,
    METADATA_DIR / f"{OUT_PREFIX}_historical_to_canonical_label_audit.tsv",
)

print("[MAIN canonical labels]")
display(
    obs["liana_celltype_main"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="n")
    .head(50)
)

print("[EXPLORATORY canonical labels]")
display(
    obs["liana_celltype_exploratory"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="n")
    .head(80)
)

print("[LABEL CORRECTION AUDIT]")
display(label_audit_summary.head(50))


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_prepare_HVG8000_r0_6_historical_to_canonical_label_audit.tsv shape=(18, 5)
[MAIN canonical labels]


,label,n
0,NaN,167178
1,Malignant_OPC_NPC_like_malignant,61646
2,Malignant_cycling_candidate,38634
3,Malignant_hypoxia_stress_malignant,32896
4,Malignant_AC_like_reactive_malignant,22525
5,Oligodendrocyte_myelinating,14809
6,Malignant_NPC_neuronal_like_malignant,6183
7,Mural_Pericyte_Fibroblast,3566
8,Endothelial,2640
9,B_Plasma,1167


[EXPLORATORY canonical labels]


,label,n
0,Malignant_OPC_NPC_like_malignant,61646
1,Review_cluster_1_review_myeloid_C1Q_TREM2_APOE...,59791
2,Malignant_cycling_candidate,38634
3,Review_cluster_3_review_myeloid_inflammatory_M...,35355
4,Malignant_hypoxia_stress_malignant,32896
5,Review_cluster_5_review_T_NK_cells_dataset_bia...,23977
6,Review_cluster_6_review_myeloid_APC_HLA_CD163_...,22943
7,Malignant_AC_like_reactive_malignant,22525
8,Oligodendrocyte_myelinating,14809
9,Review_cluster_9_review_ambiguous_mixed_Hb_T_N...,13864


[LABEL CORRECTION AUDIT]


,historical_main_label,canonical_main_label,historical_exploratory_label,canonical_exploratory_label,n_cells
2,Malignant_OPC_NPC_like_malignant,Malignant_OPC_NPC_like_malignant,Malignant_OPC_NPC_like_malignant,Malignant_OPC_NPC_like_malignant,61646
13,NaN,NaN,Review_cluster_1_review_myeloid_C1Q_TREM2_APOE...,Review_cluster_1_review_myeloid_C1Q_TREM2_APOE...,59791
3,Malignant_cycling_candidate,Malignant_cycling_candidate,Malignant_cycling_candidate,Malignant_cycling_candidate,38634
14,NaN,NaN,Review_cluster_3_review_myeloid_inflammatory_M...,Review_cluster_3_review_myeloid_inflammatory_M...,35355
4,Malignant_hypoxia_stress_malignant,Malignant_hypoxia_stress_malignant,Malignant_hypoxia_stress_malignant,Malignant_hypoxia_stress_malignant,32896
15,NaN,NaN,Review_cluster_5_review_T_NK_cells_dataset_bia...,Review_cluster_5_review_T_NK_cells_dataset_bia...,23977
16,NaN,NaN,Review_cluster_6_review_myeloid_APC_HLA_CD163_...,Review_cluster_6_review_myeloid_APC_HLA_CD163_...,22943
0,Malignant_AC_like_reactive_malignant,Malignant_AC_like_reactive_malignant,Malignant_AC_like_reactive_malignant,Malignant_AC_like_reactive_malignant,22525
8,Malignant_non_malignant_oligodendrocyte_myelin...,Oligodendrocyte_myelinating,Malignant_non_malignant_oligodendrocyte_myelin...,Oligodendrocyte_myelinating,14809
17,NaN,NaN,Review_cluster_9_review_ambiguous_mixed_Hb_T_N...,Review_cluster_9_review_ambiguous_mixed_Hb_T_N...,13864


In [7]:
# Historical execution checkpoints.
# These values come from the executed historical LIANA-preparation notebook and
# are used to ensure that the canonical reconstruction preserves the underlying
# cell inventory while correcting only label semantics.

EXPECTED_N_OBS = 351_427
EXPECTED_N_VARS = 15_176

EXPECTED_MAIN_COUNTS = {
    "Malignant_OPC_NPC_like_malignant": 61_646,
    "Malignant_cycling_candidate": 38_634,
    "Malignant_hypoxia_stress_malignant": 32_896,
    "Malignant_AC_like_reactive_malignant": 22_525,
    "Oligodendrocyte_myelinating": 14_809,
    "Malignant_NPC_neuronal_like_malignant": 6_183,
    "Mural_Pericyte_Fibroblast": 3_566,
    "Endothelial": 2_640,
    "B_Plasma": 1_167,
    "Mast_cell": 183,
}

if int(n_obs) != EXPECTED_N_OBS:
    raise RuntimeError(
        f"Unexpected cell count: observed={n_obs}, expected={EXPECTED_N_OBS}"
    )

if int(n_vars) != EXPECTED_N_VARS:
    raise RuntimeError(
        f"Unexpected gene count: observed={n_vars}, expected={EXPECTED_N_VARS}"
    )

observed_main_counts = (
    obs["liana_celltype_main"]
    .value_counts(dropna=True)
    .astype(int)
    .to_dict()
)

if observed_main_counts != EXPECTED_MAIN_COUNTS:
    expected_s = pd.Series(EXPECTED_MAIN_COUNTS, name="expected").sort_index()
    observed_s = pd.Series(observed_main_counts, name="observed").sort_index()
    comparison = pd.concat([expected_s, observed_s], axis=1)
    comparison["delta"] = comparison["observed"] - comparison["expected"]
    display(comparison)
    raise RuntimeError(
        "Canonical LIANA label counts do not match the historical cell inventory."
    )

print("PASS: historical cell/gene inventory matched.")
print("PASS: corrected main-label counts matched expected historical memberships.")


PASS: historical cell/gene inventory matched.
PASS: corrected main-label counts matched expected historical memberships.


In [8]:
# Cell type counts.
def count_table(label_col):
    d = (
        obs.dropna(subset=[label_col])
        .groupby(label_col, observed=True)
        .size()
        .reset_index(name="n_cells")
        .rename(columns={label_col: "liana_celltype"})
        .sort_values("n_cells", ascending=False)
    )
    d["analysis_label"] = label_col
    return d

main_counts = count_table("liana_celltype_main")
exploratory_counts = count_table("liana_celltype_exploratory")
celltype_counts = pd.concat([main_counts, exploratory_counts], ignore_index=True)

write_tsv_replace(celltype_counts, METADATA_DIR / f"{OUT_PREFIX}_celltype_counts.tsv")
display(celltype_counts.head(80))

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_prepare_HVG8000_r0_6_celltype_counts.tsv shape=(28, 3)


,liana_celltype,n_cells,analysis_label
0,Malignant_OPC_NPC_like_malignant,61646,liana_celltype_main
1,Malignant_cycling_candidate,38634,liana_celltype_main
2,Malignant_hypoxia_stress_malignant,32896,liana_celltype_main
3,Malignant_AC_like_reactive_malignant,22525,liana_celltype_main
4,Oligodendrocyte_myelinating,14809,liana_celltype_main
5,Malignant_NPC_neuronal_like_malignant,6183,liana_celltype_main
6,Mural_Pericyte_Fibroblast,3566,liana_celltype_main
7,Endothelial,2640,liana_celltype_main
8,B_Plasma,1167,liana_celltype_main
9,Mast_cell,183,liana_celltype_main


In [9]:
# Sample-level and condition-level counts.
def grouped_counts(label_col):
    tmp = obs.dropna(subset=[label_col]).copy()
    sample_counts = (
        tmp.groupby(["_condition", "_dataset_id", "_patient_id", "_sample_id", label_col], observed=True)
        .size()
        .reset_index(name="n_cells")
        .rename(columns={
            "_condition": "condition",
            "_dataset_id": "dataset_id",
            "_patient_id": "patient_id",
            "_sample_id": "sample_id",
            label_col: "liana_celltype",
        })
    )

    condition_counts = (
        tmp.groupby(["_condition", label_col], observed=True)
        .size()
        .reset_index(name="n_cells")
        .rename(columns={"_condition": "condition", label_col: "liana_celltype"})
    )

    return sample_counts, condition_counts

main_sample_counts, main_condition_counts = grouped_counts("liana_celltype_main")
expl_sample_counts, expl_condition_counts = grouped_counts("liana_celltype_exploratory")

main_sample_counts["analysis_mode"] = "main"
main_condition_counts["analysis_mode"] = "main"
expl_sample_counts["analysis_mode"] = "exploratory"
expl_condition_counts["analysis_mode"] = "exploratory"

sample_counts = pd.concat([main_sample_counts, expl_sample_counts], ignore_index=True)
condition_counts = pd.concat([main_condition_counts, expl_condition_counts], ignore_index=True)

write_tsv_replace(sample_counts, METADATA_DIR / f"{OUT_PREFIX}_sample_celltype_counts.tsv")
write_tsv_replace(condition_counts, METADATA_DIR / f"{OUT_PREFIX}_condition_celltype_counts.tsv")

display(condition_counts.head(80))

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_prepare_HVG8000_r0_6_sample_celltype_counts.tsv shape=(1799, 7)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_prepare_HVG8000_r0_6_condition_celltype_counts.tsv shape=(112, 4)


,condition,liana_celltype,n_cells,analysis_mode
0,GBM,B_Plasma,12,main
1,GBM,Endothelial,1026,main
2,GBM,Malignant_AC_like_reactive_malignant,13834,main
3,GBM,Malignant_NPC_neuronal_like_malignant,4423,main
4,GBM,Malignant_OPC_NPC_like_malignant,27377,main
5,GBM,Malignant_cycling_candidate,17282,main
6,GBM,Malignant_hypoxia_stress_malignant,19953,main
7,GBM,Mast_cell,4,main
8,GBM,Mural_Pericyte_Fibroblast,623,main
9,GBM,Oligodendrocyte_myelinating,9045,main


In [10]:
# Inclusion decisions for main LIANA analysis.
main_tmp = obs.dropna(subset=["liana_celltype_main"]).copy()

total_by_ct = main_tmp.groupby("liana_celltype_main", observed=True).size().rename("n_total_cells")
samples_by_ct = main_tmp.groupby("liana_celltype_main", observed=True)["_sample_id"].nunique().rename("n_samples_present")
conditions_by_ct = main_tmp.groupby("liana_celltype_main", observed=True)["_condition"].nunique().rename("n_conditions_present")

cond_pivot = (
    main_tmp.groupby(["liana_celltype_main", "_condition"], observed=True)
    .size()
    .unstack(fill_value=0)
)

decision = pd.concat([total_by_ct, samples_by_ct, conditions_by_ct], axis=1).reset_index().rename(columns={"liana_celltype_main": "liana_celltype"})
for cond in sorted(obs["_condition"].unique()):
    if cond in cond_pivot.columns:
        decision[f"n_cells_{cond}"] = decision["liana_celltype"].map(cond_pivot[cond]).fillna(0).astype(int)
    else:
        decision[f"n_cells_{cond}"] = 0

decision["include_main_global"] = (
    (decision["n_total_cells"] >= MIN_TOTAL_CELLS_MAIN) &
    (decision["n_samples_present"] >= MIN_SAMPLES_MAIN)
)

cond_cols = [c for c in decision.columns if c.startswith("n_cells_")]
decision["n_conditions_with_min_cells"] = (decision[cond_cols] >= MIN_CELLS_PER_CONDITION_FOR_CONDITION_ANALYSIS).sum(axis=1)
decision["include_condition_specific"] = decision["n_conditions_with_min_cells"] >= 2

decision["note"] = np.where(
    decision["include_main_global"],
    "included in main LIANA global analysis",
    "excluded from main LIANA global analysis due to low cells or low sample support"
)

decision = decision.sort_values(["include_main_global", "n_total_cells"], ascending=[False, False])
write_tsv_replace(decision, METADATA_DIR / f"{OUT_PREFIX}_celltype_inclusion_decisions.tsv")
display(decision)

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_prepare_HVG8000_r0_6_celltype_inclusion_decisions.tsv shape=(10, 12)


,liana_celltype,n_total_cells,n_samples_present,n_conditions_present,n_cells_GBM,n_cells_LGG,n_cells_ndGBM,n_cells_rGBM,include_main_global,n_conditions_with_min_cells,include_condition_specific,note
4,Malignant_OPC_NPC_like_malignant,61646,79,4,27377,3975,18830,11464,True,4,True,included in main LIANA global analysis
5,Malignant_cycling_candidate,38634,75,4,17282,283,10730,10339,True,4,True,included in main LIANA global analysis
6,Malignant_hypoxia_stress_malignant,32896,75,4,19953,38,9476,3429,True,3,True,included in main LIANA global analysis
2,Malignant_AC_like_reactive_malignant,22525,80,4,13834,507,4864,3320,True,4,True,included in main LIANA global analysis
9,Oligodendrocyte_myelinating,14809,72,4,9045,1180,1241,3343,True,4,True,included in main LIANA global analysis
3,Malignant_NPC_neuronal_like_malignant,6183,66,4,4423,220,1241,299,True,4,True,included in main LIANA global analysis
8,Mural_Pericyte_Fibroblast,3566,60,4,623,36,2446,461,True,3,True,included in main LIANA global analysis
1,Endothelial,2640,62,4,1026,168,1002,444,True,4,True,included in main LIANA global analysis
0,B_Plasma,1167,41,4,12,2,619,534,True,2,True,included in main LIANA global analysis
7,Mast_cell,183,12,4,4,3,122,54,False,2,True,excluded from main LIANA global analysis due t...


In [11]:
# Export per-cell metadata for next LIANA notebook.
cols_to_export = [
    "_cell_id",
    "_condition",
    "_dataset_id",
    "_patient_id",
    "_sample_id",
    "_cluster",
    "_annotation",
    "_compartment",
    "_malignant_state",
    "_copykat",
    "liana_celltype_main",
    "liana_celltype_exploratory",
]

cell_meta = obs[cols_to_export].copy()
cell_meta = cell_meta.rename(columns={
    "_cell_id": "cell_id",
    "_condition": "condition",
    "_dataset_id": "dataset_id",
    "_patient_id": "patient_id",
    "_sample_id": "sample_id",
    "_cluster": "cluster",
    "_annotation": "final_annotation_refined",
    "_compartment": "final_compartment_major",
    "_malignant_state": "malignant_state",
    "_copykat": "copykat_pred_clean",
})

included_main_types = set(decision.loc[decision["include_main_global"], "liana_celltype"])
cell_meta["liana_include_main_global"] = cell_meta["liana_celltype_main"].isin(included_main_types)
cell_meta["liana_include_exploratory"] = cell_meta["liana_celltype_exploratory"].notna()

write_tsv_replace(cell_meta, METADATA_DIR / f"{OUT_PREFIX}_cell_metadata.tsv")

main_cells = cell_meta[cell_meta["liana_include_main_global"]].copy()
expl_cells = cell_meta[cell_meta["liana_include_exploratory"]].copy()

write_tsv_replace(main_cells, METADATA_DIR / f"{OUT_PREFIX}_main_cells.tsv")
write_tsv_replace(expl_cells, METADATA_DIR / f"{OUT_PREFIX}_exploratory_cells.tsv")

print("[main cells]", main_cells.shape)
print("[exploratory cells]", expl_cells.shape)
display(main_cells.head())

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_prepare_HVG8000_r0_6_cell_metadata.tsv shape=(351427, 14)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_prepare_HVG8000_r0_6_main_cells.tsv shape=(184066, 14)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_prepare_HVG8000_r0_6_exploratory_cells.tsv shape=(351427, 14)
[main cells] (184066, 14)
[exploratory cells] (351427, 14)


,cell_id,condition,dataset_id,patient_id,sample_id,cluster,final_annotation_refined,final_compartment_major,malignant_state,copykat_pred_clean,liana_celltype_main,liana_celltype_exploratory,liana_include_main_global,liana_include_exploratory
DS1_GSE103224:PJ016_cell000216,DS1_GSE103224:PJ016_cell000216,GBM,DS1,PJ016,PJ016,7,malignant_AC_like_reactive_CNV_supported,malignant_like,AC_like_reactive_malignant,diploid,Malignant_AC_like_reactive_malignant,Malignant_AC_like_reactive_malignant,True,True
DS1_GSE103224:PJ016_cell000217,DS1_GSE103224:PJ016_cell000217,GBM,DS1,PJ016,PJ016,7,malignant_AC_like_reactive_CNV_supported,malignant_like,AC_like_reactive_malignant,diploid,Malignant_AC_like_reactive_malignant,Malignant_AC_like_reactive_malignant,True,True
DS1_GSE103224:PJ016_cell000218,DS1_GSE103224:PJ016_cell000218,GBM,DS1,PJ016,PJ016,7,malignant_AC_like_reactive_CNV_supported,malignant_like,AC_like_reactive_malignant,diploid,Malignant_AC_like_reactive_malignant,Malignant_AC_like_reactive_malignant,True,True
DS1_GSE103224:PJ016_cell000219,DS1_GSE103224:PJ016_cell000219,GBM,DS1,PJ016,PJ016,2,cycling_malignant_like_candidate_CNV_supported,cycling_malignant_like_candidate,cycling_malignant_candidate,diploid,Malignant_cycling_candidate,Malignant_cycling_candidate,True,True
DS1_GSE103224:PJ016_cell000220,DS1_GSE103224:PJ016_cell000220,GBM,DS1,PJ016,PJ016,7,malignant_AC_like_reactive_CNV_supported,malignant_like,AC_like_reactive_malignant,diploid,Malignant_AC_like_reactive_malignant,Malignant_AC_like_reactive_malignant,True,True


In [12]:
# Readiness and manifest.
readiness = pd.DataFrame([
    {"item": "input_h5ad", "value": str(INPUT_H5AD), "status": "found"},
    {"item": "n_cells", "value": int(n_obs), "status": "ok"},
    {"item": "n_genes", "value": int(n_vars), "status": "ok"},
    {"item": "cluster_col", "value": cluster_col, "status": "ok"},
    {"item": "annotation_col", "value": annotation_col, "status": "ok"},
    {"item": "compartment_col", "value": compartment_col, "status": "ok"},
    {"item": "condition_col", "value": condition_col, "status": "ok"},
    {"item": "sample_col", "value": sample_col, "status": "ok"},
    {"item": "patient_col", "value": patient_col if patient_col else "sample_id used as patient_id", "status": "ok"},
    {"item": "dataset_col", "value": dataset_col if dataset_col else "unknown_dataset", "status": "ok"},
    {"item": "copykat_col", "value": copykat_col if copykat_col else "not_available", "status": "ok"},
    {"item": "malignant_state_col", "value": malignant_state_col, "status": "ok"},
    {"item": "historical_label_bug", "value": "non_malignant substring was captured by malignant branch", "status": "corrected"},
    {"item": "label_correction_membership_change", "value": False, "status": "verified"},
    {"item": "main_liana_celltypes", "value": int(main_counts.shape[0]), "status": "ok"},
    {"item": "main_cells_included_global", "value": int(main_cells.shape[0]), "status": "ok"},
    {"item": "exploratory_cells", "value": int(expl_cells.shape[0]), "status": "ok"},
    {"item": "write_duplicate_liana_ready_h5ad", "value": bool(WRITE_LIANA_READY_H5AD), "status": "configured"},
    {"item": "created_at", "value": datetime.now().isoformat(timespec="seconds"), "status": "ok"},
])

readiness_path = METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv"
write_tsv_replace(readiness, readiness_path)
display(readiness)

manifest_records = []
for p in [
    METADATA_DIR / f"{OUT_PREFIX}_resolved_columns.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_historical_to_canonical_label_audit.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_celltype_counts.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_sample_celltype_counts.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_condition_celltype_counts.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_celltype_inclusion_decisions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_cell_metadata.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_main_cells.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_exploratory_cells.tsv",
    readiness_path,
]:
    manifest_records.append({
        "path": str(p),
        "exists": p.exists(),
        "size_mb": round(p.stat().st_size / 1024 / 1024, 3) if p.exists() else np.nan,
    })

manifest = pd.DataFrame(manifest_records)
manifest_path = METADATA_DIR / f"{OUT_PREFIX}_output_manifest.tsv"
write_tsv_replace(manifest, manifest_path)
display(manifest)

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_prepare_HVG8000_r0_6_readiness.tsv shape=(19, 3)


,item,value,status
0,input_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...,found
1,n_cells,351427,ok
2,n_genes,15176,ok
3,cluster_col,leiden_scvi_main,ok
4,annotation_col,final_annotation_refined,ok
5,compartment_col,final_compartment_major,ok
6,condition_col,condition,ok
7,sample_col,sample_id,ok
8,patient_col,patient_id,ok
9,dataset_col,dataset_id,ok


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_prepare_HVG8000_r0_6_output_manifest.tsv shape=(10, 3)


,path,exists,size_mb
0,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
1,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.003
2,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.002
3,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.148
4,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.006
5,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
6,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,85.034
7,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,45.386
8,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,85.034
9,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001


## What to send back after running this notebook

Please send these files after a successful fresh-kernel run:

```text
metadata/LIANA_prepare_HVG8000_r0_6_readiness.tsv
metadata/LIANA_prepare_HVG8000_r0_6_resolved_columns.tsv
metadata/LIANA_prepare_HVG8000_r0_6_historical_to_canonical_label_audit.tsv
metadata/LIANA_prepare_HVG8000_r0_6_celltype_counts.tsv
metadata/LIANA_prepare_HVG8000_r0_6_sample_celltype_counts.tsv
metadata/LIANA_prepare_HVG8000_r0_6_condition_celltype_counts.tsv
metadata/LIANA_prepare_HVG8000_r0_6_celltype_inclusion_decisions.tsv
metadata/LIANA_prepare_HVG8000_r0_6_output_manifest.tsv
```

Do **not** send the full h5ad.
